In [ ]:
!pip install medmnist==3.0.2
!pip install torch==2.10.0
!pip install torchvision==0.25.0
!pip install numpy==2.4.3

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
import medmnist
from medmnist import INFO

import numpy as np
import random

In [2]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

# **Exp 1**

## **Downloading the Dataset**

BloodMNIST is a medical dataset consisting of **17,092** grayscale-equivalent RGB images (28x28 pixels) of **8 different types of blood cells** (e.g., Lymphocytes, Monocytes, Platelets).


In [3]:
data_flag = 'bloodmnist'
info = INFO[data_flag]
DataClass = getattr(medmnist, info['python_class'])

In [4]:
# we got the DataClass for the dataset, for the sake of this assignment, we will flat the images with small preprocessing

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    transforms.Lambda(lambda x: x.view(-1))   # to flat the images
])

train_dataset = DataClass(split='train', transform=transform, download=True)
validation_dataset = DataClass(split='val', transform=transform, download=True)
test_dataset = DataClass(split= 'test', transform=transform, download=True)

train_loader = DataLoader(dataset=train_dataset, batch_size=128, shuffle=False)
val_loader = DataLoader(dataset=validation_dataset, batch_size=128, shuffle=False)
test_loader = DataLoader(dataset=test_dataset, batch_size=128)


print(f"The image size is {train_dataset.size ** 2 * train_dataset.info['n_channels']}")
print(f"The number of samples is {train_dataset.info['n_samples']}")
print(f"The number of classes is {len(train_dataset.info['label'])}")


100%|██████████| 35.5M/35.5M [00:02<00:00, 12.3MB/s]


The image size is 2352
The number of samples is {'train': 11959, 'val': 1712, 'test': 3421}
The number of classes is 8


## **Defining the model and training structure**

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"The current device is {device}")

The current device is cuda


In [70]:
# we are going to define a class for our architecture (it will dynamic to try different depth and many options)

class BloodMNISTModel(nn.Module):

    def __init__(self, layers: list = []):
        super().__init__()
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

    @classmethod
    def create_one_layer_model(cls, input_size: int = 2352, num_classes: int = 8):
        return cls([nn.Linear(input_size, num_classes)])

    @classmethod
    def create_complex_model_without_regularization(cls, input_size: int = 2352, num_classes: int = 8):
        layers = [
            nn.Linear(input_size, 2048), nn.ReLU(),
            nn.Linear(2048, 1024), nn.ReLU(),
            nn.Linear(1024,1024), nn.ReLU(),
            nn.Linear(1024, 512), nn.ReLU(),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 64), nn.ReLU(),
            nn.Linear(64, num_classes)
        ]
        return cls(layers)

    @classmethod
    def create_complex_model_with_regularization(cls, input_size: int = 2352, num_classes: int = 8):
        layers = [
            nn.Linear(input_size, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),


            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
          ]
        return cls(layers)



In [34]:
def train_one_epoch(model, train_loader, optimizer, criterion, sanity_data=None):
    model.train()

    if sanity_data is not None:
        images, labels = sanity_data
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        return loss.item()

    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device).long().squeeze()
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(train_loader)

def eval_model(model, val_loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device).long().squeeze()

            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    avg_loss = total_loss / len(val_loader)
    accuracy = 100 * correct / total

    return avg_loss, accuracy



## **Step 1: Sanity Check**

In [8]:
# We will use 4 data points for our sanity check

data_iter = iter(train_loader)
fixed_images, fixed_labels = next(data_iter)
fixed_images = fixed_images[:4].to(device)
fixed_labels = fixed_labels[:4].to(device).long().squeeze()

In [9]:
epochs = 100

model = BloodMNISTModel.create_one_layer_model()     # define our one layer model
model.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [10]:
for epoch in range(epochs):
    loss = train_one_epoch(model, train_loader, optimizer, criterion, sanity_data=(fixed_images, fixed_labels))
    if epoch % 10 == 0:
        print(f"Sanity Epoch {epoch} | Loss: {loss:.6f}")

Sanity Epoch 0 | Loss: 2.142314
Sanity Epoch 10 | Loss: 0.039291
Sanity Epoch 20 | Loss: 0.005658
Sanity Epoch 30 | Loss: 0.002517
Sanity Epoch 40 | Loss: 0.001664
Sanity Epoch 50 | Loss: 0.001366
Sanity Epoch 60 | Loss: 0.001234
Sanity Epoch 70 | Loss: 0.001154
Sanity Epoch 80 | Loss: 0.001092
Sanity Epoch 90 | Loss: 0.001037


In [11]:
# test it on test dataset

test_loss, test_accuracy = eval_model(model, test_loader, criterion)
print(f"Test Loss: {test_loss:.6f} | Test Accuracy: {test_accuracy:.2f}%")

Test Loss: 8.358374 | Test Accuracy: 34.96%


## **Step 2: Train a simple model**

In [12]:
epochs = 100

model = BloodMNISTModel.create_one_layer_model()     # define our one layer model
model.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [13]:
for epoch in range(epochs):
    loss = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_accuracy = eval_model(model, val_loader, criterion)

    if epoch % 10 == 0:
        print(f"Training Epoch {epoch} | Loss: {loss:.6f}")
        print(f"Validation Epoch {epoch} | Loss {val_loss:.6f} | Accuracy {val_accuracy:.6f}")

Training Epoch 0 | Loss: 1.028848
Validation Epoch 0 | Loss 0.761570 | Accuracy 75.525701
Training Epoch 10 | Loss: 0.561355
Validation Epoch 10 | Loss 0.545876 | Accuracy 81.308411
Training Epoch 20 | Loss: 0.492961
Validation Epoch 20 | Loss 0.511276 | Accuracy 82.242991
Training Epoch 30 | Loss: 0.453588
Validation Epoch 30 | Loss 0.493957 | Accuracy 82.535047
Training Epoch 40 | Loss: 0.426356
Validation Epoch 40 | Loss 0.483454 | Accuracy 82.301402
Training Epoch 50 | Loss: 0.406105
Validation Epoch 50 | Loss 0.477888 | Accuracy 82.476636
Training Epoch 60 | Loss: 0.390408
Validation Epoch 60 | Loss 0.476440 | Accuracy 83.294393
Training Epoch 70 | Loss: 0.377837
Validation Epoch 70 | Loss 0.478354 | Accuracy 83.119159
Training Epoch 80 | Loss: 0.367467
Validation Epoch 80 | Loss 0.482611 | Accuracy 82.885514
Training Epoch 90 | Loss: 0.358686
Validation Epoch 90 | Loss 0.487980 | Accuracy 82.710280


In [14]:
# test it on test dataset

test_loss, test_accuracy = eval_model(model, test_loader, criterion)
print(f"Test Loss: {test_loss:.6f} | Test Accuracy: {test_accuracy:.2f}%")

Test Loss: 0.522679 | Test Accuracy: 81.41%


## **Step 3: Train a complex model (wish to overfit)**

In [52]:
epochs = 100

model = BloodMNISTModel.create_complex_model_without_regularization()     # define our complex model
model.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [53]:
for epoch in range(epochs):
    loss = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_accuracy = eval_model(model, val_loader, criterion)

    if epoch % 10 == 0:
        print(f"Training Epoch {epoch} | Loss: {loss:.6f}")
        print(f"Validation Epoch {epoch} | Loss {val_loss:.6f} | Accuracy {val_accuracy:.6f}")

Training Epoch 0 | Loss: 1.392159
Validation Epoch 0 | Loss 0.831992 | Accuracy 66.822430
Training Epoch 10 | Loss: 0.418350
Validation Epoch 10 | Loss 0.475863 | Accuracy 82.768692
Training Epoch 20 | Loss: 0.261162
Validation Epoch 20 | Loss 0.463801 | Accuracy 85.747664
Training Epoch 30 | Loss: 0.207476
Validation Epoch 30 | Loss 0.599618 | Accuracy 82.418224
Training Epoch 40 | Loss: 0.133499
Validation Epoch 40 | Loss 0.608587 | Accuracy 86.448598
Training Epoch 50 | Loss: 0.103934
Validation Epoch 50 | Loss 0.808433 | Accuracy 84.929907
Training Epoch 60 | Loss: 0.086951
Validation Epoch 60 | Loss 0.685174 | Accuracy 87.091121
Training Epoch 70 | Loss: 0.053058
Validation Epoch 70 | Loss 0.845210 | Accuracy 87.908879
Training Epoch 80 | Loss: 0.048055
Validation Epoch 80 | Loss 0.866313 | Accuracy 86.857477
Training Epoch 90 | Loss: 0.044590
Validation Epoch 90 | Loss 0.754823 | Accuracy 88.960280


In [55]:
# test it on test dataset

test_loss, test_accuracy = eval_model(model, test_loader, criterion)
print(f"Test Loss: {test_loss:.6f} | Test Accuracy: {test_accuracy:.2f}%")

Test Loss: 1.012132 | Test Accuracy: 86.52%


## **Step 4: Train a complex model with regularization**

In [71]:
epochs = 100

model = BloodMNISTModel.create_complex_model_with_regularization()     # define our complex model
model.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [72]:
for epoch in range(epochs):
    loss = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_accuracy = eval_model(model, val_loader, criterion)

    if epoch % 10 == 0:
        print(f"Training Epoch {epoch} | Loss: {loss:.6f}")
        print(f"Validation Epoch {epoch} | Loss {val_loss:.6f} | Accuracy {val_accuracy:.6f}")

Training Epoch 0 | Loss: 0.953753
Validation Epoch 0 | Loss 0.691010 | Accuracy 72.663551
Training Epoch 10 | Loss: 0.233725
Validation Epoch 10 | Loss 0.423504 | Accuracy 85.572430
Training Epoch 20 | Loss: 0.137932
Validation Epoch 20 | Loss 0.532005 | Accuracy 84.929907
Training Epoch 30 | Loss: 0.087194
Validation Epoch 30 | Loss 0.417648 | Accuracy 88.434579
Training Epoch 40 | Loss: 0.065284
Validation Epoch 40 | Loss 0.453451 | Accuracy 88.668224
Training Epoch 50 | Loss: 0.053414
Validation Epoch 50 | Loss 0.537962 | Accuracy 87.908879
Training Epoch 60 | Loss: 0.041613
Validation Epoch 60 | Loss 0.523727 | Accuracy 88.317757
Training Epoch 70 | Loss: 0.035940
Validation Epoch 70 | Loss 0.451930 | Accuracy 89.602804
Training Epoch 80 | Loss: 0.037098
Validation Epoch 80 | Loss 0.533418 | Accuracy 89.778037
Training Epoch 90 | Loss: 0.028514
Validation Epoch 90 | Loss 0.536041 | Accuracy 89.485981


In [74]:
# test it on test dataset

test_loss, test_accuracy = eval_model(model, test_loader, criterion)
print(f"Test Loss: {test_loss:.6f} | Test Accuracy: {test_accuracy:.2f}%")

Test Loss: 0.595802 | Test Accuracy: 88.42%


I tried various model architectures, I tried to overfit the training data and it did but it got a reasonable validation accuracy and test accuracy.

After the regularization, it just improved a little, it still overfits but got a tiny bit better validation and test accuracy. I think maybe the data problem isn't that bad so these models get stuck at some point around 88%.

# **Exp 2**

## **Downloading the dataset**

In [75]:
data_flag = 'bloodmnist'
info = INFO[data_flag]
DataClass = getattr(medmnist, info['python_class'])

In [78]:

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

train_dataset = DataClass(split='train', transform=transform, download=True)
validation_dataset = DataClass(split='val', transform=transform, download=True)
test_dataset = DataClass(split= 'test', transform=transform, download=True)

train_loader = DataLoader(dataset=train_dataset, batch_size=128, shuffle=False)
val_loader = DataLoader(dataset=validation_dataset, batch_size=128, shuffle=False)
test_loader = DataLoader(dataset=test_dataset, batch_size=128)


print(f"The image size is {train_dataset.size}×{train_dataset.size}×{train_dataset.info['n_channels']}")
print(f"The number of samples is {train_dataset.info['n_samples']}")
print(f"The number of classes is {len(train_dataset.info['label'])}")


The image size is 28×28×3
The number of samples is {'train': 11959, 'val': 1712, 'test': 3421}
The number of classes is 8


## **Setting up the Model**

In [79]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"The current device is {device}")

The current device is cuda


In [85]:
class BloodMNIST_CNN(nn.Module):
    def __init__(self, layers: list):
        super().__init__()
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

    @classmethod
    def create_one_layer_model(cls, num_classes: int = 8):
        layers = [
            nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(16 * 14 * 14, num_classes)
        ]
        return cls(layers)

    @classmethod
    def create_complex_without_regularization(cls, num_classes: int = 8):
        layers = [
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 512), nn.ReLU(),
            nn.Linear(512, num_classes)
        ]
        return cls(layers)

    @classmethod
    def create_complex_model_with_regularization(cls, num_classes: int = 8):
        layers = [

            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.2),

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.3),

            # Final classifier
            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        ]
        return cls(layers)

In [86]:
def train_one_epoch(model, train_loader, optimizer, criterion, sanity_data=None):
    model.train()

    if sanity_data is not None:
        images, labels = sanity_data
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        return loss.item()

    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device).long().squeeze()
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(train_loader)

def eval_model(model, val_loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device).long().squeeze()

            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    avg_loss = total_loss / len(val_loader)
    accuracy = 100 * correct / total

    return avg_loss, accuracy



## **Step 1**

In [87]:
# We will use 4 data points for our sanity check

data_iter = iter(train_loader)
fixed_images, fixed_labels = next(data_iter)
fixed_images = fixed_images[:4].to(device)
fixed_labels = fixed_labels[:4].to(device).long().squeeze()

In [89]:
epochs = 100

model = BloodMNIST_CNN.create_one_layer_model()     # define our one layer model
model.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [90]:
for epoch in range(epochs):
    loss = train_one_epoch(model, train_loader, optimizer, criterion, sanity_data=(fixed_images, fixed_labels))
    if epoch % 10 == 0:
        print(f"Sanity Epoch {epoch} | Loss: {loss:.6f}")

Sanity Epoch 0 | Loss: 2.133126
Sanity Epoch 10 | Loss: 0.241745
Sanity Epoch 20 | Loss: 0.031595
Sanity Epoch 30 | Loss: 0.007594
Sanity Epoch 40 | Loss: 0.003252
Sanity Epoch 50 | Loss: 0.002095
Sanity Epoch 60 | Loss: 0.001606
Sanity Epoch 70 | Loss: 0.001338
Sanity Epoch 80 | Loss: 0.001157
Sanity Epoch 90 | Loss: 0.001011


In [91]:
# test it on test dataset

test_loss, test_accuracy = eval_model(model, test_loader, criterion)
print(f"Test Loss: {test_loss:.6f} | Test Accuracy: {test_accuracy:.2f}%")

Test Loss: 8.529091 | Test Accuracy: 36.42%


## **Step 2**

In [94]:
epochs = 100

model = BloodMNIST_CNN.create_one_layer_model()     # define our one layer model
model.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [95]:
for epoch in range(epochs):
    loss = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_accuracy = eval_model(model, val_loader, criterion)

    if epoch % 10 == 0:
        print(f"Training Epoch {epoch} | Loss: {loss:.6f}")
        print(f"Validation Epoch {epoch} | Loss {val_loss:.6f} | Accuracy {val_accuracy:.6f}")

Training Epoch 0 | Loss: 1.042872
Validation Epoch 0 | Loss 0.650979 | Accuracy 79.264019
Training Epoch 10 | Loss: 0.277024
Validation Epoch 10 | Loss 0.320340 | Accuracy 88.726636
Training Epoch 20 | Loss: 0.188667
Validation Epoch 20 | Loss 0.283079 | Accuracy 90.303738
Training Epoch 30 | Loss: 0.139397
Validation Epoch 30 | Loss 0.281043 | Accuracy 90.654206
Training Epoch 40 | Loss: 0.103326
Validation Epoch 40 | Loss 0.293623 | Accuracy 90.595794
Training Epoch 50 | Loss: 0.076068
Validation Epoch 50 | Loss 0.314948 | Accuracy 90.186916
Training Epoch 60 | Loss: 0.056974
Validation Epoch 60 | Loss 0.350901 | Accuracy 90.070093
Training Epoch 70 | Loss: 0.049610
Validation Epoch 70 | Loss 0.380075 | Accuracy 89.953271
Training Epoch 80 | Loss: 0.048346
Validation Epoch 80 | Loss 0.365885 | Accuracy 91.179907
Training Epoch 90 | Loss: 0.032737
Validation Epoch 90 | Loss 0.427674 | Accuracy 90.186916


In [96]:
# test it on test dataset

test_loss, test_accuracy = eval_model(model, test_loader, criterion)
print(f"Test Loss: {test_loss:.6f} | Test Accuracy: {test_accuracy:.2f}%")

Test Loss: 0.489842 | Test Accuracy: 89.86%


## **Step 3**

In [97]:
epochs = 100

model = BloodMNIST_CNN.create_complex_without_regularization()     # define our complex model
model.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [98]:
for epoch in range(epochs):
    loss = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_accuracy = eval_model(model, val_loader, criterion)

    if epoch % 10 == 0:
        print(f"Training Epoch {epoch} | Loss: {loss:.6f}")
        print(f"Validation Epoch {epoch} | Loss {val_loss:.6f} | Accuracy {val_accuracy:.6f}")

Training Epoch 0 | Loss: 1.057372
Validation Epoch 0 | Loss 0.658401 | Accuracy 74.474299
Training Epoch 10 | Loss: 0.109243
Validation Epoch 10 | Loss 0.258209 | Accuracy 92.523364
Training Epoch 20 | Loss: 0.043413
Validation Epoch 20 | Loss 0.288648 | Accuracy 92.932243
Training Epoch 30 | Loss: 0.013985
Validation Epoch 30 | Loss 0.354375 | Accuracy 93.983645
Training Epoch 40 | Loss: 0.019284
Validation Epoch 40 | Loss 0.521360 | Accuracy 92.348131
Training Epoch 50 | Loss: 0.008355
Validation Epoch 50 | Loss 0.334578 | Accuracy 94.392523
Training Epoch 60 | Loss: 0.007063
Validation Epoch 60 | Loss 0.364017 | Accuracy 94.801402
Training Epoch 70 | Loss: 0.000014
Validation Epoch 70 | Loss 0.423966 | Accuracy 94.334112
Training Epoch 80 | Loss: 0.000006
Validation Epoch 80 | Loss 0.440166 | Accuracy 94.392523
Training Epoch 90 | Loss: 0.000003
Validation Epoch 90 | Loss 0.454985 | Accuracy 94.275701


In [99]:
# test it on test dataset

test_loss, test_accuracy = eval_model(model, test_loader, criterion)
print(f"Test Loss: {test_loss:.6f} | Test Accuracy: {test_accuracy:.2f}%")

Test Loss: 0.522458 | Test Accuracy: 94.01%


## **Step 4**

In [100]:
epochs = 100

model = BloodMNIST_CNN.create_complex_model_with_regularization()     # define our complex model with regularization
model.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [101]:
for epoch in range(epochs):
    loss = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_accuracy = eval_model(model, val_loader, criterion)

    if epoch % 10 == 0:
        print(f"Training Epoch {epoch} | Loss: {loss:.6f}")
        print(f"Validation Epoch {epoch} | Loss {val_loss:.6f} | Accuracy {val_accuracy:.6f}")

Training Epoch 0 | Loss: 1.042910
Validation Epoch 0 | Loss 0.477729 | Accuracy 81.892523
Training Epoch 10 | Loss: 0.272654
Validation Epoch 10 | Loss 0.204601 | Accuracy 93.224299
Training Epoch 20 | Loss: 0.201843
Validation Epoch 20 | Loss 0.197471 | Accuracy 94.042056
Training Epoch 30 | Loss: 0.133506
Validation Epoch 30 | Loss 0.174702 | Accuracy 94.275701
Training Epoch 40 | Loss: 0.103835
Validation Epoch 40 | Loss 0.166724 | Accuracy 94.976636
Training Epoch 50 | Loss: 0.088818
Validation Epoch 50 | Loss 0.169895 | Accuracy 95.560748
Training Epoch 60 | Loss: 0.061858
Validation Epoch 60 | Loss 0.204745 | Accuracy 94.801402
Training Epoch 70 | Loss: 0.053755
Validation Epoch 70 | Loss 0.199105 | Accuracy 95.443925
Training Epoch 80 | Loss: 0.042190
Validation Epoch 80 | Loss 0.200489 | Accuracy 95.502336
Training Epoch 90 | Loss: 0.047193
Validation Epoch 90 | Loss 0.238377 | Accuracy 94.801402


In [102]:
# test it on test dataset

test_loss, test_accuracy = eval_model(model, test_loader, criterion)
print(f"Test Loss: {test_loss:.6f} | Test Accuracy: {test_accuracy:.2f}%")

Test Loss: 0.252402 | Test Accuracy: 94.94%


Here I tried with the CNN models,  the simple model got a reasonable accuracy, the complex model overfitted the training data badly in terms of loss, but the accuracy was still not bad.

After regularization, the complex model got a bit better but within same range of accuracy, but the gap in loss is smaller than before.